In [ ]:
# SPDX-License-Identifier: GPL-3.0-or-later

# Workflow to perform full-field recrystallization simulations

This notebook describes the workflow that is described in the paper **Open-source simulation and research data management tools combined for studying recrystallization in industrial aluminium alloys**

The workflow here follows figure 1 in the paper. 

<div class="alert alert-info">
This notebook is an example how the software tools in the paper were combined and how these tools generated the individual data artifacts. To assure that the example executes quickly even on modest hardware you should consider the following:
</div>

1. The RVE volume considered here is smaller.
2. The deformation level applied here is smaller. Consequently, fewer remeshing steps are required. Specifically, two deformation steps with 0.1 compressive strain each are simulated. Between these two deformation steps, there is a remeshing step. For the sake of readability, the code blocks for running the simulations are shortened by using helper functions.
3. The source code to evaluate the nucleation criterion that is described in the paper is not included in the SCORE version as that code is proprietary.

In [ ]:
import os, sys
import shutil
import json

import h5py
import numpy as np
import subprocess
from jupyterlab_h5web import H5Web

cwd_path = os.getcwd()
print(cwd_path)
# sys.path.append(f"{cwd_path}/scripts/")

## Synthetic microstructure generation (DREAM3D)

The microstructure parameters used to create the synthetic microstructure are described in Section 2.2.2 and 2.2.4 in the paper.

The average grain size (ESD) compares to the measured grain size (ECD). The average precipitate size (ESD) also compares to the measured precipitate size (ECD). Moreover, the volume fraction of the second phase corresponds to the area fraction of the second phase measured.

This section details the legacy `DREAM3D` pipeline that is pre-designed to create a representative volume element (RVE) of a two phase microstructure. This volume is discretized using 15 cells each along of the principal directions (the rolling, the transverse, and the normal direction). The voxel size is 0.8 micrometer. In case of changing the microstructure generation settings, please look at the documentation of [DREAM3D](https://www.dream3d.io/python_docs/index.html).

<div class="alert alert-info">
The DREAM3D project has been in a process of substantial refactoring while we worked on this paper. This affected our work in all steps.
</div>

The original `DREAM3D` software is now considered legacy. Version [`v6.5.171`](https://github.com/BlueQuartzSoftware/DREAM3D/releases) is the last official release. Legacy `DREAM3D` is no longer maintained. Instead, the Bluequartz developers are maintaining and evolving the followed up software product called [`DREAM3D-NX`](https://www.dream3d.io/). As of May 6th, 2026 its official version was `v7.4.1`. The functionalities of legacy `DREAM3D` and `DREAM3D-NX` differ.

    
Firstly, the functionalities for synthesizing microstructures have not yet been ported into `DREAM3D-NX`. Therefore, we here show and advise to continue to use legacy `DREAM3D`. We exemplify in this workflow to use said `v6.5.171`. Specifically, we use the `PipelineRunner` functionality enabling programmatic execution. Note that we motivated the [`H5Web`](https://h5web.panosc.eu/) developers to include the DREAM3D specific mime type ending `.dream3d` in their whitelist of recognized HDF5 file types, enabling displaying of legacy DREAM3D files directly in a jupyter notebook via [`jupyterlab-h5web`](https://github.com/silx-kit/jupyterlab-h5web) which we install via its conda-forge package.

Secondly, the functionalities for characterizing and processing orientation analyses on synthetic microstructures, may these originate from post-processed measurements or from simulations like ours, have been ported into `DREAM3D-NX`. Therefore, we here show an example how to programmatically run these functionalities using `DREAM3D-NX`. Specifically, we use the `simplnx` library as it is offered by the bluequartzsoftware `conda` channel via its `dream3dnx` `conda` package.

In [ ]:
import simplnx as nx
dream3d_bin_path = f"../../thirdparty/dream3d/DREAM3D-6.5.171-Linux-x86_64/bin"
print(f"Legacy DREAM3D location\n{dream3d_bin_path}")

pipeline_file_path = f"{cwd_path}/Microstructure_for_DAMASK.json"
print(f"Pipeline file location\n{pipeline_file_path}")

In [ ]:
cmd = f"{dream3d_bin_path}/PipelineRunner -p {pipeline_file_path}"
p = subprocess.Popen(cmd, shell=True)                                                                    
while p.poll() == None:
    p.poll()                                                                               
if p.poll() == 0:                                                                                                       
   print(f'Synthetic microstructure created!!')

In [ ]:
H5Web("Microstructure_for_DAMASK.dream3d")

## Crystal plasticity simulation (DAMASK)

### Import DREAM3D file and parameterize the DAMASK simulation

<div class="alert alert-warning">
Hi Vitesh, please have all files stored right in `examples/amag`.
</div>

In [ ]:
from damask import GeomGrid
from damask import ConfigMaterial
import damask

Create the RVE geometry file how to discretize the RVE into material points.

In [ ]:
geom = GeomGrid.load_DREAM3D(
    f"{cwd_path}/Microstructure_for_DAMASK.dream3d"
    # feature_IDs="/DataContainers/SyntheticVolumeDataContainer/CellData"
)

In [ ]:
geom.size = geom.size * 1E-06

In [ ]:
geom.save(f"Microstructure_for_DAMASK.vti")
geom

Create the material file

In [ ]:
mat = ConfigMaterial.load_DREAM3D(
    f"Microstructure_for_DAMASK.dream3d",
    base_group="/DataContainers/SyntheticVolumeDataContainer"
    # grain_data="/DataContainers/SyntheticVolumeDataContainer/Grain Data"
)

mat.save(f"material.yaml")

The material file created at this step has only the grain orientation and the assigned phase for each material point.
We define the material parameters for each of these phases.

In [ ]:
mat = ConfigMaterial.load(f"material.yaml")

example_mat = ConfigMaterial.load(f"{cwd_path}/example_material.yaml")

mat['phase']['Aluminum'] = example_mat['phase']['Aluminum']
mat['phase']['Second_phase'] = example_mat['phase']['Second_phase']
mat['homogenization']['direct'] = example_mat['homogenization']['SX']

mat.save(f"material.yaml")

<div class="alert alert-warning">
Hi Vitesh, please work on this part, its the DAMASK one.
</div>

### Run the first deformation step (DAMASK)

In [ ]:
geom_file = f"Microstructure_for_DAMASK"
load_file = f"tensionX"
mat_file = f"material"
max_processes = 1
# max_threads = int(os.cpu_count()/2 if os.cpu_count() >= 2 else 1)
print(f"Running with {max_processes} MPI processes")
# ! export OMP_NUM_THREADS=$max_threads
! echo $OMP_NUM_THREADS

In [ ]:
cmd = f"mpiexec -n {max_processes} DAMASK_grid -m {mat_file}.yaml -l {load_file}.yaml -g {geom_file}.vti"

In [ ]:
damask.util.run(cmd, wd=simulation_path)

In [ ]:
# add H5Web to inspect HDF5 file

### Run the first regridding (Python)

In [ ]:
import damask_regridding as rg
from damask_to_dream3d import DAMASKtoDREAM3D

<div class="alert alert-warning">
Please note that I have moved the damask specific scripts to src/python
Please let us not do os.chdir but rather also work from the perspective of `cwd_path`.
</div>

In [ ]:
# os.chdir(simulation_folder)

In [ ]:
map_0to_rg, cells_rg, size_rg, increment_title = rg.regrid_geom(geom_file,load_file)
rg.write_geomRegridded(geom_file,increment_title,map_0to_rg,cells_rg,size_rg)
rg.write_h5OutRestart(geom_file,load_file,increment_title,map_0to_rg,cells_rg,isElastic=False)

Adding another load step of 0.1 compression strain

In [ ]:
# os.chdir(simulation_folder)
rg.add_load_step(0.0076,3000,1000,1000)

 Preparing files for next deformation step in DAMASK

In [ ]:
copy(f'{geom_file}_regridded_{increment_title}.vti',f'{geom_file}.vti')
copy(f'{geom_file}_{load_file}_restart_regridded_{increment_title}.hdf5',f'{geom_file}_{load_file}_restart.hdf5')

<div class="alert alert-warning">
I am not at all a fan of replacing files once they are generated, once generate files should always remain immutable, otherwise provenance checking is super tricky
Would it be possible such fine run the second deformation step from the geom_file_regridded_increment_title.vti file?
</div>

### Run the second deformation step (DAMASK)

In [ ]:
restart_inc = increment_title.split('increment_')[1]
cmd = f'mpiexec -n {proc} DAMASK_grid -l {load_file}.yaml -g {geom_file}.vti -r {restart_inc}'

In [ ]:
damask.util.run(f'mpiexec -n {proc} DAMASK_grid -l {load_file}.yaml -g {geom_file}.vti -r {restart_inc}', wd=simulation_folder)

In [ ]:
# add H5Web to inspect HDF5 file

### Run the second regridding (Python)

In [ ]:
map_0to_rg, cells_rg, size_rg, increment_title = rg.regrid_geom(geom_file,load_file)
rg.write_geomRegridded(geom_file,increment_title,map_0to_rg,cells_rg,size_rg)
rg.write_h5OutRestart(geom_file,load_file,increment_title,map_0to_rg,cells_rg,isElastic=False)

copy(f'{geom_file}_regridded_{increment_title}.vti',f'{geom_file}.vti')
copy(f'{geom_file}_{load_file}_restart_regridded_{increment_title}.hdf5',f'{geom_file}_{load_file}_restart.hdf5')

restart_inc = increment_title.split('increment_')[1]

## Post-processing DAMASK data to SCORE

In [ ]:
# os.chdir(simulation_folder)
d = damask.Result(f'{geom_file}_{load_file}.hdf5')
d.add_calculation('np.sum(#rho_mob#,axis=1) + np.sum(#rho_dip#,axis=1)','tot_density')          # calculating the total dislocation density

Adding the necessary post-processed datasets to the file.

In [ ]:
d = DAMASKtoDREAM3D(simulation_folder,f'{geom_file}_{load_file}.hdf5',geom_file,load_file)
d.DAMASKtoDREAM3D_deformed(int(restart_inc))
d.export_DREAM3D(int(restart_inc),'tot_density')

Remeshing the file for `SCORE` as `SCORE` needs a regularly spaced grid the following call yields such regularly spaced grid.

In [ ]:
d.remesh_DREAM3D(int(restart_inc))

In [ ]:
# add H5Web to inspect file

### Parameterize the nucleation model for SCORE.

In [ ]:
Parameterize the threshold dislocation density (here 3E15 m ** -2) above which nucleation sites are chosen.

In [ ]:
d.add_nucleation_max_rho(int(restart_inc), 3E15)

In [ ]:
# add H5Web to inspect file

### Microstructure transfer between DAMASK and SCORE

In [ ]:
pipeline_template_file = f"{cwd_path}/Microstructure_for_SCORE_template.json"
# we generate an instance of the template as now we know the finally processed geometry from DAMASK
pipeline_file = f"{cwd_path}/example_files/Microstructure_for_SCORE_instance.json"

geom_file_name = "Microstructure_for_DAMASK"
load_file_name = "tensionX"
restart_inc = 6000

In [ ]:
# we have still somewhere a mixture between dream3d and dream3D let's use dream3d all throughout

In [ ]:
with h5py.File(f"{simulation_path}/{geom_file_name}_{load_file_name}_remeshed_increment_{restart_inc}.dream3d") as h5r:
    tuple_dimensions = np.array(h5r['/DataContainers/SyntheticVolumeDataContainer/_SIMPL_GEOMETRY/DIMENSIONS']).tolist()
    voxel_spacing = np.array(h5r['/DataContainers/SyntheticVolumeDataContainer/_SIMPL_GEOMETRY/SPACING']).tolist()
tuple_dimensions, voxel_spacing

In [ ]:
with open(pipeline_template_file) as fp:
    data = json.load(fp)

    # create an instance from the template
    data['0']['InputFile'] = f"{simulation_path}/{geom_file_name}_{load_file_name}_remeshed_increment_{restart_inc}.dream3d"
    data['0']['InputFileDataContainerArrayProxy']['Data Containers'][0]['Attribute Matrices'][0]['Data Arrays'][0]['Tuple Dimensions'] = tuple_dimensions
    data['0']['InputFileDataContainerArrayProxy']['Data Containers'][0]['Attribute Matrices'][0]['Data Arrays'][1]['Tuple Dimensions'] = tuple_dimensions
    data['0']['InputFileDataContainerArrayProxy']['Data Containers'][0]['Attribute Matrices'][0]['Data Arrays'][2]['Tuple Dimensions'] = tuple_dimensions
    data['8']['OutputFile'] = f"{simulation_path}/{geom_file_name}_{load_file_name}_remeshed_increment_{restart_inc}_segmented.dream3d"

with open(pipeline_file, 'w') as fp:
    json.dump(data,fp,indent=4)

In [ ]:
"""
os.chdir(dream_3d_bin_path)                                                                                             
cmd = f'./PipelineRunner -p {dream_3d_pipeline_file}'                                                                   
p = subprocess.Popen(cmd,shell=True)                                                                                    
while p.poll() == None:
    p.poll()                                                                                                                                                                                                                                     
if p.poll() == 0:                                                                                                       
   print(f'Microstructure segmented. Run SCORE!!')
"""

<div class="alert alert-warning">
Handover, Vitesh again to me
</div>

## Recrystallization simulation (SCORE)

### Parameterize the recrystallization simulation (Python)

In [ ]:
from pint import Quantity
from score_config import ScoreConfig
from score_config_utils import ureg

In [ ]:
MY_MATERIAL: list[tuple[str, Quantity]] = [
    (
        "melting_temperature",
        ureg.Quantity(np.float64(650.0), ureg.degC).to(ureg.kelvin),
    ),
    (
        "shear_modulus_zero",
        ureg.Quantity(np.float64(26.0), ureg.gigapascal).to(ureg.pascal),
    ),
    ("nadal_lepoac_a", ureg.Quantity(np.float64(0.469))),
    ("nadal_lepoac_zeta", ureg.Quantity(np.float64(0.04))),
    ("burgers_vector_zero", ureg.Quantity(np.float64(0.286), ureg.nanometer)),
    # shown here how to consider a constant Burgers vector
    ("lattice_expansion_second", ureg.Quantity(0.0, (ureg.kelvin**-2))),
    ("lattice_expansion_first", ureg.Quantity(0.0, (ureg.kelvin**-1))),
    ("lattice_expansion_null", ureg.Quantity(0.0, ureg.nanometer)),
]

MY_USER: list[str] = [
    "Vitesh Shah",
    "Markus Kühbach",
    "Sebastian Samberger",
    "Georg Falkinger",
    "Franz Roters",
]

MY_INIT_DAMASK: dict[str, str] = {
    "file_path": f"{os.getcwd()}/../../examples/damask/Tphase_Vf_01_tensionX_remeshed_increment_19000_segmented.dream3d"
}

MY_MOBILITY_RH: list[tuple[str, Quantity]] = [
    (
        "m_null",
        ureg.Quantity(np.float64(300.0), ureg.meter**4 / (ureg.joule * ureg.second)),
    ),
    ("enthalpy", ureg.Quantity(np.float64(2.0314), ureg.eV)),
    # ("c1", ureg.Quantity(np.float64(0.9))),
    # ("c2", ureg.Quantity(np.float64(5.))),
    # ("c3", ureg.Quantity(np.float64(9.))),
]

MY_TIME_TEMPERATURE: list[tuple[str, Quantity]] = [
    (
        "time",
        ureg.Quantity(
            np.asarray(
                [
                    0.0,
                    5.8,
                    100.8,
                    103.7,
                    105.0,
                    200.8,
                    205.8,
                    208.7,
                    300.0,
                    305.8,
                    405.8,
                    408.7,
                    600.0,
                ],
                np.float64,
            ),
            ureg.second,
        ),
    ),
    (
        "temperature",
        ureg.Quantity(
            np.asarray(
                [
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                ],
                np.float64,
            ),
            ureg.degC,
        ).to(ureg.kelvin),
    ),
]

MY_GRID_CUBOIDAL: list[tuple[str, Quantity]] = [
    ("extent", ureg.Quantity(np.asarray([19, 15, 11], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.5), ureg.micrometer)),
]

MY_GRID_DAMASK: list[tuple[str, Quantity]] = [
    ("extent", ureg.Quantity(np.asarray([128, 52, 64], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.4982421), ureg.micrometer)),
]

In [ ]:
print("Generating a NeXus/HDF5 configuration file for examples/amag")
nxs = ScoreConfig(1)
nxs.user(MY_USER)
nxs.sample("Al")
nxs.material(MY_MATERIAL)
nxs.deformation("damask", MY_INIT_DAMASK)
nxs.nucleation("damask")
nxs.mobility("rollett_holm", MY_MOBILITY_RH)
nxs.time_temperature(MY_TIME_TEMPERATURE)
nxs.discretization(MY_GRID_DAMASK)
nxs.numerics()
nxs.get_config()
nxs.write_config_hfive_file()

In [ ]:
H5Web(f"{cwd_path}/SCORE.Config.SimID.{nxs.sim_id}.nxs")

### Run the recrystallization simulation (SCORE)

In [ ]:
# currently pointing to the wrong inputs, just for testing
score_config_file = f"{cwd_path}/SCORE.Config.SimID.{nxs.sim_id}.nxs"

In [ ]:
cmd = f"mpiexec -n 1 ./../../build/score {nxs.sim_id} {score_config_file}"
p = subprocess.Popen(cmd, shell=True)
while p.poll() == None:
    p.poll()                                                                                                                                                                                                                                     
if p.poll() == 0:                                                                                                       
   print(f'Recrystallization simulation completed')

## Inspect the simulation results (e.g. H5Web)

In [ ]:
H5Web(f"{cwd_path}/SCORE.Results.SimID.{nxs.sim_id}.nxs")

***
This completes the tutorial.